# Algoritmo Genético

A estrutura do algoritmo conssiste em alguns passos:

1. Geracão de populacão de indivíduos.
2. Selecão de indivíduos:
   1. Aptidão
   2. Sorteio
   3. Combinacão de ambos
3. Cruzamento entre indivíduos selecionados
4. Mutacão

## Casos para testar

Pensando na dinâmica do algoritmo, para garantir a funcionalidade, foi pensado algumas funcões para testar as capabilidades do algoritmo:

1. $f(x) = -(x-3)^2 + 10$
2. $f(x) = -x^2$
3. $f(x) = 10x^2 - x^4$
4. $f(x) = \frac{x}{(x^2 + 1)}$

# Ambiente

Bibliotecas, módulos e processo básico para settar o documento.

In [1]:
import random, math

## Parâmetros do algoritmo

Eu pensei em criar parâmetros para inicializacão das variáveis, então o mínimo e o máximo são relativos aos pontos de partida, quando inicializarmos a populacão.
Além disso, os parâmetros além deles são relativos a configuracão da populacão em si.

In [2]:
# Relativos somente a definicão da populacão

LIMITE_INFERIOR = -512
LIMITE_SUPERIOR = 512

# Definicões gerais

TAMANHO_POPULACAO = 100
NUM_GERACOES = 100
REDUCAO_POR_GERACAO = 0.9

TAXA_CROSSOVER = 0.8
TAXA_MUTACAO = 0.1

# Funcões para avaliar

# funcão 1 (baseada no doc original):
FUNCAO = lambda x: (-(x-3)**2)+10

# funcão 2:
# FUNCAO = lambda x: (-(x)**2)

# funcão 3:
# FUNCAO = lambda x: (10(x**2) - x**4)

# funcão 4:
# FUNCAO = lambda x: (x) / (x**2 + 1)


# 1. Geração de população de indivíduos

Nessa implementação, consideramos o indivíduo como valores de X a serem passados como parâmetro para a própria função.

O indivíduo nasce a partir de um valor aleatório dentro do limite superior ou inferior, mas pode passar ou não dele.

In [3]:
def criar_individuo():
    individuo = random.randint(LIMITE_INFERIOR, LIMITE_SUPERIOR)
    return individuo

Para a populacão, seria estender para vários indivíduos.

In [4]:
def criar_populacao(n):
    return [criar_individuo() for _ in range(n)]

populacao_atual = TAMANHO_POPULACAO

# 2. Selecão de indivíduos:

Foram montados alguns processos para selecão, como podem ser usados funcões com múltiplos possíveis picos, por exemplo, a estratégia aplicada envolvia selecionar dentro da populacão vários espacos e desses espacos, procurar o máximo.

### Calcular aptidão:

Máximo:

Simplesmente a própria função sendo aplicada ao valor do indivíduo. Como o objetivo é maximizar a função, basta testar o valor de X e ver se o Y aumenta ou diminui.

In [62]:
def fitness(funcao, individuo):
    return funcao(individuo)

Mínimo:

Para encontrar o mínimo de uma função, existem algumas metodologias. Algumas delas:


*   Mais simples - negativar a função:
    *   Um potencial problema é que indivíduos potencialmente bons podem dominar a seleção, a depender do método usado, fazendo o algoritmo chegar em respostas próximas, mas não necessariamente corretas.

$$
\ fitness(x) = -f(x)
$$

*   Inversão direta somando 1 no denominador - valores menores produzindo maior fitness:
    *   Essa é uma estratégia pouco robusta contra outliers, quando o *y* se aproxima de 0 ou assume valores grandes.

$$
\ fitness(x) = \frac{1}{1+f(x)}
$$

*   Ranking inverso: usa o mesmo fitness de maximizar, mas na hora da seleção, inverte o ranking.
    *   Essa estratégia pode ser muito útil, porém um de seus problemas é que ela descarta a diferença entre os indivíduos após ranqueá-los. Uma possibilidade é usar ranking + elitismo para reduzir esse impacto.

In [ ]:
def fitness_min(funcao, individuo):
    return -funcao(individuo)

In [ ]:
def fitness_min(funcao, individuo):
    return 1 / (1 + funcao(individuo))

## Selecões:

Para garantir opcões, montei algumas opcões:

1. Aleatória + Aptidão (já feita no documento)
2. Intervalos Regulares + Aptidão

#### 1. Aleatória + Aptidão


Aqui optei por uma abordagem mista: primeiro seleciono alguns indivíduos aleatoriamente e depois pego o mais apto entre eles. Uma abordagem não-elitista.

Isso pode ser bom e ruim, na pior das hipóteses você acaba escolhendo os piores para se analisar, mas garante que não sigamos para um falso melhor, tenhamos sempre opcões em pontos distintos, eu também implementei algumas outras opcões.

In [6]:
def selecao_aleatoria_aptidao(populacao, funcao, verbose=False):
    selecionados = []
    qtd_geracao = int((1 - REDUCAO_POR_GERACAO) * populacao_atual)
    
    if verbose:
        print(f"qtd_geracao: {qtd_geracao}")
        
    for _ in range(qtd_geracao):
        indice_selecionado = random.randint(0, len(populacao)-1)
        individuo = populacao[indice_selecionado]
        
        if verbose:
            print(f"Indivíduo selecionado aleatoriamente: {individuo}")
            print(f"Fitness: {fitness(funcao, individuo)}")
            print()

        selecionados.append(
            (individuo, fitness(funcao, individuo))
        )
    
    return max(selecionados, key=lambda x: x[1])[0]

#### 2. Intervalos Regulares + Aptidão

Aqui a ideia surgiu de montar intervalos regulares dentro do qual poderiam ser selecionados os máximos, o comportamento não garante algo sem falhas, mas garante que vai testar opcões em pontos mais distribuídos dentre os dados recebidos.

In [7]:
def selecao_intervalo_aptidao(populacao, funcao, verbose=False):
    selecionados = []
    qtd_geracao = int((1 - REDUCAO_POR_GERACAO) * populacao_atual)
    
    qtd_analise = populacao_atual // qtd_geracao
    if verbose:
        print(f"qtd_geracao: {qtd_geracao} | qtd_analise: {qtd_analise}")
    
    for intervalo in range(0, populacao_atual, qtd_analise):
        
        indice_apto_intervalo = max(
            populacao[intervalo:intervalo + qtd_analise], 
            key=lambda x: fitness(funcao, x)
        )
        individuo_apto = populacao[indice_apto_intervalo]
        
        if verbose:
            print(f"Dentre as opcões: {populacao[intervalo:intervalo + qtd_analise]}")
            print(f"Selecinado: {individuo_apto} | Porque fitness: {fitness(funcao, individuo_apto)}")
            print()
        
        selecionados.append(
            (individuo_apto, fitness(funcao, individuo_apto))
        )
    
    return max(selecionados, key=lambda x: x[1])[0]

### Testes para a selecão

Testes de execucão básica para os dois algoritmos

In [28]:
dados_teste = list(range(populacao_atual))
print(f"Dados para o teste: {dados_teste}")
print("=" * 50)
funcoes_possiveis = [
    selecao_aleatoria_aptidao,
    selecao_intervalo_aptidao
]

for funcao in funcoes_possiveis:
    print(f"Para a selecão {funcao.__name__}:")
    
    resultado = funcao(dados_teste, FUNCAO, verbose=True)
    
    print(f"Resultado final: {resultado} | Fitness: {fitness(FUNCAO, resultado)}")
    print("-" * 50)
    print()

Dados para o teste: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Para a selecão selecao_aleatoria_aptidao:
qtd_geracao: 9
Indivíduo selecionado aleatoriamente: 19
Fitness: -246

Indivíduo selecionado aleatoriamente: 97
Fitness: -8826

Indivíduo selecionado aleatoriamente: 55
Fitness: -2694

Indivíduo selecionado aleatoriamente: 45
Fitness: -1754

Indivíduo selecionado aleatoriamente: 24
Fitness: -431

Indivíduo selecionado aleatoriamente: 32
Fitness: -831

Indivíduo selecionado aleatoriamente: 40
Fitness: -1359

Indivíduo selecionado aleatoriamente: 23
Fitness: -390

Indivíduo selecionado aleatoriamente: 53
Fitness: -2490

Resultado final: 19 | F

# 3. Cruzamento entre indivíduos selecionados

No documento anterior era executado de forma binária. Eu preferi não fazer daquele jeito porque ficaria definido pelo nosso intervalo, poderíamos errar para onde prosseguir e estariamos limitando o modelo a nosso viés.

Sobre como vou aproximar esse problema, o crossover pode comecar de qualquer ponto, não necessariamente do mesmo ponto toda vez, para ser mais semelhante a característica do genoma, eu segui esse princípio.

Para isso eu me baseei [nessa](https://www.geeksforgeeks.org/machine-learning/crossover-in-genetic-algorithm/) página do geeks for geeks.
Mas tem [esse](https://www.youtube.com/watch?v=BTlB6ioeMSU) vídeo de indiano que citou MUITAS opcões.

Pensei em algumas maneiras de aproximar esse problema:
1. Ponto único
2. Dois pontos
3. Uniforme

#### 1. Ponto único

A ideia é definir um ponto do qual dali em diante o código será proveniente de outro pai. Esse ponto será aleatório.

In [19]:
def crossover_single_point(pai1, pai2, test=False):
    if random.random() < TAXA_CROSSOVER or test:
        
        pai1_bin = bin(pai1)[2:]
        pai2_bin = bin(pai2)[2:]
        
        maior_tamanho = max(len(pai1_bin), len(pai2_bin))
        pai1_bin = pai1_bin.zfill(maior_tamanho)
        pai2_bin = pai2_bin.zfill(maior_tamanho)
        
        ponto = random.randint(0, maior_tamanho - 1)
        
        x_filho1 = pai1_bin[:ponto] + pai2_bin[ponto:]
        x_filho2 = pai2_bin[:ponto] + pai1_bin[ponto:]

        filho1 = int(x_filho1, 2)
        filho2 = int(x_filho2, 2)

        if test: 
            print(f"Ponto a partir do qual vai ter o merge: {ponto}")
            print(f"filho1: {filho1} = {x_filho1} | filho2: {filho2} = {x_filho2}")

        return filho1, filho2

    else:
        return pai1, pai2

#### 2. Dois pontos
Vamos definir um intervalo para a mesclagem, como as sequências podem ter tamanhos distintos, vamos escolher por base na de menor tamanho.


In [ ]:
def crossover_dual_point(pai1, pai2, test=False):
    if random.random() < TAXA_CROSSOVER or test:
        
        pai1_bin = bin(pai1)[2:]
        pai2_bin = bin(pai2)[2:]
        
        maior_tamanho = max(len(pai1_bin), len(pai2_bin))
        
        pai1_bin = pai1_bin.zfill(maior_tamanho)
        pai2_bin = pai2_bin.zfill(maior_tamanho)
        
        ponto_ini = random.randint(0, maior_tamanho - 2)
        ponto_fim = random.randint(ponto_ini, maior_tamanho - 1)
        
        x_filho1 = pai1_bin[:ponto_ini] + pai2_bin[ponto_ini:ponto_fim] + pai1_bin[ponto_fim:]
        x_filho2 = pai2_bin[:ponto_ini] + pai1_bin[ponto_ini:ponto_fim] + pai2_bin[ponto_fim:]
        
        filho1 = int(x_filho1, 2)
        filho2 = int(x_filho2, 2)
        
        if test: 
            print(f"Ponto a partir do qual vai ter o merge: {ponto_ini}")
            print(f"Ponto até onde irá ter o merge: {ponto_fim}")
            
            print(f"filho1: {filho1} = {x_filho1} | filho2: {filho2} = {x_filho2}")

        return filho1, filho2

    else:
        return pai1, pai2

#### 3. Uniforme 

Modifica ponto a ponto de forma aleatória, não é só um intervalo, a cada ponto avalia a possibilidade de cruzar com o outro.

In [ ]:
def crossover_uniform(pai1, pai2, test=False):
    if random.random() < TAXA_CROSSOVER or test:
        
        pai1_bin = bin(pai1)[2:]
        pai2_bin = bin(pai2)[2:]
        
        maior_tamanho = max(len(pai1_bin), len(pai2_bin))
        
        pai1_bin = pai1_bin.zfill(maior_tamanho)
        pai2_bin = pai2_bin.zfill(maior_tamanho)
                
        x_filho1 = ""
        x_filho2 = ""
        
        for indice in range(maior_tamanho):
            mantem = bool(random.randint(0, 1))
            
            if mantem and test:
                print(f"Índice {indice} houve troca.")
            
            x_filho1 += pai1_bin[indice] if mantem else pai2_bin[indice]
            x_filho2 += pai2_bin[indice] if mantem else pai1_bin[indice]
            
        
        filho1 = int(x_filho1, 2)
        filho2 = int(x_filho2, 2)

        if test: 
            print(f"filho1: {filho1} = {x_filho1} | filho2: {filho2} = {x_filho2}")
        
        return filho1, filho2

    else:
        return pai1, pai2

### Teste para Crossover
Testes para avaliar como o crossover tem funcionado.

In [69]:
pai1 = 123
pai2 = 456

print(f"Pais assumidos para o teste:\n - {pai1} = {bin(pai1)[2:]}\n - {pai2} = {bin(pai2)[2:]}")
print("=" * 50)

funcoes_possiveis = [
    crossover_single_point,
    crossover_dual_point,
    crossover_uniform
]

for funcao in funcoes_possiveis:
    print(f"Para a selecão {funcao.__name__}:")
    
    resultado = funcao(pai1, pai2, test=True)
    print("-" * 50)
    print()

Pais assumidos para o teste:
 - 123 = 1111011
 - 456 = 111001000
Para a selecão crossover_single_point:
pai1_bin: 001111011 | pai2_bin: 111001000
Ponto a partir do qual vai ter o merge: 7
filho1: 120 = 001111000 | filho2: 459 = 111001011
--------------------------------------------------

Para a selecão crossover_dual_point:
pai1_bin: 001111011 | pai2_bin: 111001000
Ponto a partir do qual vai ter o merge: 2
Ponto até onde irá ter o merge: 7
filho1: 75 = 001001011 | filho2: 504 = 111111000
--------------------------------------------------

Para a selecão crossover_uniform:
Índice 4 houve troca.
Índice 5 houve troca.
Índice 6 houve troca.
Índice 7 houve troca.
Índice 8 houve troca.
filho1: 475 = 111011011 | filho2: 104 = 001101000
--------------------------------------------------



# 4. Mutação

Tem uma pequena chance de ocorrer uma mutação, e um bit aleatório de um indivíduo da população ser alterado.

In [53]:
def mutar(individuo, test=False):
    if random.random() < TAXA_MUTACAO or test:
        x_individuo = list(bin(individuo)[2:])
        indice_mudar = random.randint(0, len(x_individuo) - 1)
        
        x_individuo[indice_mudar] = str(1 - int(x_individuo[indice_mudar]))
        novos_bits = "".join(x_individuo)
        
        if test:
            print(f"Mudando o índice {indice_mudar}: {bin(individuo)[2:]}")
            print(f"Resultado: {novos_bits} = {int(novos_bits, 2)}")
        
        return int(novos_bits, 2)

    return individuo

### Teste para Mutacão
Testes para avaliar como o mutacão tem funcionado.

In [54]:
individuo = 255

print(f"Para o teste da funcão \"mutar\":")
print("-" * 50)
resultado = mutar(individuo, test=True)


Para o teste da funcão "mutar":
--------------------------------------------------
Mudando o índice 6: 11111111
Resultado: 11111101 = 253
